# 01 · Bronze layer

First stage of the pipeline: load the raw Arabic hotel reviews (`data/raw/balanced-reviews.txt`) **as-is** with PySpark, attach ingestion metadata, and persist them as Parquet in `data/bronze/`.

No cleaning happens here — bronze is a faithful, typed copy of the source so later layers (silver → gold → AraBERT fine-tuning) can always be rebuilt from it.

| | |
|---|---|
| **Input** | `data/raw/balanced-reviews.txt` (tab-separated, UTF-16) |
| **Output** | `data/bronze/reviews.parquet` |

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

## Spark session

A local session using all CPU cores (`local[*]`) — enough for a dataset of this size on one machine.

In [ ]:
spark = (SparkSession.builder
        .appName("BronzeLayer")
        .master("local[*]")
        .getOrCreate())

## Schema

An explicit schema instead of `inferSchema`: faster (no extra pass over the file), predictable types, and clean `snake_case` column names. Columns are matched by position, so the source header (`no`, `Hotel name`, …) is renamed here.

- `nights` stays a string — values are Arabic phrases (e.g. `أقمت ليلة واحدة`), parsed later in silver.
- `_corrupt_record` captures any line that doesn't fit the schema instead of failing the job.

In [ ]:
schema = T.StructType([
    T.StructField("review_id", T.IntegerType(), True),
    T.StructField("hotel_name", T.StringType(), True),
    T.StructField("rating", T.IntegerType(), True),
    T.StructField("user_type", T.StringType(), True),
    T.StructField("room_type", T.StringType(), True),
    T.StructField("nights", T.StringType(), True),
    T.StructField("review", T.StringType(), True),
    T.StructField("_corrupt_record", T.StringType())]
)

## Read the raw file

- `sep="\t"`, `encoding="UTF-16"` — the source is a UTF-16 tab-separated file.
- `multiline=True` — tolerates reviews that contain line breaks.
- `mode="PERMISSIVE"` — malformed rows are kept and routed to `_corrupt_record` rather than dropped silently.

In [ ]:
raw = (spark.read
        .option("header", True)
        .option("sep", "\t")
        .option("encoding", "UTF-16")
        .option("multiline", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .csv("../data/raw/balanced-reviews.txt"))

## Add ingestion metadata

Lineage columns (prefixed `_` to mark them as pipeline-added, not source data):

- `_ingested_at` — when this load ran.
- `_source_file` — which file each row came from, via Spark's hidden `_metadata` column.

In [ ]:
bronze = (raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path")))

## Sanity checks

Row count and number of corrupt records — the load is trustworthy only if corrupt rows are 0 (or explained). `cache()` keeps the DataFrame in memory so these actions don't re-read the file.

Last run: **105,698 rows, 0 corrupt** — matches the pandas count in `notebook/00_inspect.ipynb`.

In [ ]:
bronze.cache()
print("Rows:", bronze.count())
print("Corrupt: ", bronze.filter(F.col("_corrupt_record").isNotNull()).count())
bronze.show(truncate=False)

## Write bronze

Saved as Parquet (columnar, compressed, keeps the schema) for the silver step.

> **Note:** written via pandas/pyarrow because Spark's native writer needs Hadoop's `winutils.exe` on Windows. Once that's set up, this can become `bronze.write.mode("overwrite").parquet("../data/bronze/reviews")`.

In [ ]:
bronze.toPandas().to_parquet("../data/bronze/reviews.parquet", index=False)